# Fraud Detection Performance Comparison
## Milestone 3 – Algorithm Implementation & Analysis

---

###Project Overview
This notebook implements and evaluates two classification algorithms — **Decision Tree** and **K-Nearest Neighbors (KNN)** — for detecting fraudulent credit card transactions.

**Dataset:** Credit Card Fraud Detection 2023 (Kaggle, ~550 000 transactions)  
**Features:** `id`, anonymised features `V1–V28`, `Amount`, `Class` (0 = Normal, 1 = Fraud)

### Notebook Structure
1. [Imports & Configuration](#1-imports--configuration)
2. [Data Loading & Preprocessing](#2-data-loading--preprocessing)
3. [Helper Functions](#3-helper-functions)
4. [Algorithm 1 – Decision Tree](#4-algorithm-1--decision-tree)
5. [Algorithm 2 – K-Nearest Neighbors](#5-algorithm-2--k-nearest-neighbors)
6. [Running Time Experiments](#6-running-time-experiments)

## 1. Imports & Configuration

In [ ]:
import math
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── Reproducibility ──────────────────────────────────────────────────────
RANDOM_STATE  = 42
TEST_SIZE     = 0.3
KNN_NEIGHBORS = 5
REPEATS       = 20

SMALL_SIZES = [1000, 2000, 3000, 4000]
LARGE_SIZES = [10000, 20000, 30000, 50000]

## 2. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv("creditcard_2023.csv")

print("Shape:", df.shape)
df.head()

Shape: (568630, 31)


,id,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-0.260648,-0.469648,2.496266,-0.083724,0.129681,0.732898,0.519014,-0.130006,0.727159,...,-0.110552,0.217606,-0.134794,0.165959,0.126280,-0.434824,-0.081230,-0.151045,17982.10,0
1,1,0.985100,-0.356045,0.558056,-0.429654,0.277140,0.428605,0.406466,-0.133118,0.347452,...,-0.194936,-0.605761,0.079469,-0.577395,0.190090,0.296503,-0.248052,-0.064512,6531.37,0
2,2,-0.260272,-0.949385,1.728538,-0.457986,0.074062,1.419481,0.743511,-0.095576,-0.261297,...,-0.005020,0.702906,0.945045,-1.154666,-0.605564,-0.312895,-0.300258,-0.244718,2513.54,0
3,3,-0.152152,-0.508959,1.746840,-1.090178,0.249486,1.143312,0.518269,-0.065130,-0.205698,...,-0.146927,-0.038212,-0.214048,-1.893131,1.003963,-0.515950,-0.165316,0.048424,5384.44,0
4,4,-0.206820,-0.165280,1.527053,-0.448293,0.106125,0.530549,0.658849,-0.212660,1.049921,...,-0.106984,0.729727,-0.161666,0.312561,-0.414116,1.071126,0.023712,0.419117,14278.97,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568630 entries, 0 to 568629
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   id      568630 non-null  int64  
 1   V1      568630 non-null  float64
 2   V2      568630 non-null  float64
 3   V3      568630 non-null  float64
 4   V4      568630 non-null  float64
 5   V5      568630 non-null  float64
 6   V6      568630 non-null  float64
 7   V7      568630 non-null  float64
 8   V8      568630 non-null  float64
 9   V9      568630 non-null  float64
 10  V10     568630 non-null  float64
 11  V11     568630 non-null  float64
 12  V12     568630 non-null  float64
 13  V13     568630 non-null  float64
 14  V14     568630 non-null  float64
 15  V15     568630 non-null  float64
 16  V16     568630 non-null  float64
 17  V17     568630 non-null  float64
 18  V18     568630 non-null  float64
 19  V19     568630 non-null  float64
 20  V20     568630 non-null  float64
 21  V21     56

In [ ]:
df.describe()

,id,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,568630.000000,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,...,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,5.686300e+05,568630.000000,568630.0
mean,284314.500000,-5.638058e-17,-1.319545e-16,-3.518788e-17,-2.879008e-17,7.997245e-18,-3.958636e-17,-3.198898e-17,2.109273e-17,3.998623e-17,...,4.758361e-17,3.948640e-18,6.194741e-18,-2.799036e-18,-3.178905e-17,-7.497417e-18,-3.598760e-17,2.609101e-17,12041.957635,0.5
std,164149.486121,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,...,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,6919.644449,0.5
min,0.000000,-3.495584e+00,-4.996657e+01,-3.183760e+00,-4.951222e+00,-9.952786e+00,-2.111111e+01,-4.351839e+00,-1.075634e+01,-3.751919e+00,...,-1.938252e+01,-7.734798e+00,-3.029545e+01,-4.067968e+00,-1.361263e+01,-8.226969e+00,-1.049863e+01,-3.903524e+01,50.010000,0.0
25%,142157.250000,-5.652859e-01,-4.866777e-01,-6.492987e-01,-6.560203e-01,-2.934955e-01,-4.458712e-01,-2.835329e-01,-1.922572e-01,-5.687446e-01,...,-1.664408e-01,-4.904892e-01,-2.376289e-01,-6.515801e-01,-5.541485e-01,-6.318948e-01,-3.049607e-01,-2.318783e-01,6054.892500,0.0
50%,284314.500000,-9.363846e-02,-1.358939e-01,3.528579e-04,-7.376152e-02,8.108788e-02,7.871758e-02,2.333659e-01,-1.145242e-01,9.252647e-02,...,-3.743065e-02,-2.732881e-02,-5.968903e-02,1.590123e-02,-8.193162e-03,-1.189208e-02,-1.729111e-01,-1.392973e-02,12030.150000,0.5
75%,426471.750000,8.326582e-01,3.435552e-01,6.285380e-01,7.070047e-01,4.397368e-01,4.977881e-01,5.259548e-01,4.729905e-02,5.592621e-01,...,1.479787e-01,4.638817e-01,1.557153e-01,7.007374e-01,5.500147e-01,6.728879e-01,3.340230e-01,4.095903e-01,18036.330000,1.0
max,568629.000000,2.229046e+00,4.361865e+00,1.412583e+01,3.201536e+00,4.271689e+01,2.616840e+01,2.178730e+02,5.958040e+00,2.027006e+01,...,8.087080e+00,1.263251e+01,3.170763e+01,1.296564e+01,1.462151e+01,5.623285e+00,1.132311e+02,7.725594e+01,24039.930000,1.0


In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("\nClass distribution:")
print(df['Class'].value_counts())

Missing values:
 id        0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

Duplicate rows: 0

Class distribution:
Class
0    284315
1    284315
Name: count, dtype: int64


## 3. Helper Functions

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────

def print_evaluation(title: str, y_true, y_pred) -> None:
    """Print accuracy, confusion matrix, and classification report."""
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))


# ── Dataset Sampling ──────────────────────────────────────────────────────

def sample_balanced(data: pd.DataFrame, n: int, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """Return a class-balanced sample of size n (n/2 per class)."""
    half = n // 2
    df0 = data[data['Class'] == 0].sample(n=half,     random_state=random_state, replace=True)
    df1 = data[data['Class'] == 1].sample(n=n - half, random_state=random_state, replace=True)
    return pd.concat([df0, df1]).sample(frac=1, random_state=random_state)


def sample_imbalanced(data: pd.DataFrame, n: int, fraud_ratio: float = 0.10,
                      random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """Return an imbalanced sample (default 10 % fraud, 90 % normal)."""
    fraud_count  = max(1, int(n * fraud_ratio))
    normal_count = n - fraud_count
    df0 = data[data['Class'] == 0].sample(n=normal_count, random_state=random_state, replace=True)
    df1 = data[data['Class'] == 1].sample(n=fraud_count,  random_state=random_state, replace=True)
    return pd.concat([df0, df1]).sample(frac=1, random_state=random_state)


# ── KNN (custom implementation) ───────────────────────────────────────────

def euclidean_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """Compute the Euclidean distance between two vectors."""
    return np.sqrt(np.sum((x1 - x2) ** 2))


def knn_predict(X_train: pd.DataFrame, y_train: pd.Series,
                X_test: pd.DataFrame, k: int = KNN_NEIGHBORS) -> list:
    """
    Predict class labels for X_test using a brute-force KNN.

    Parameters
    ----------
    X_train : training feature matrix
    y_train : training labels
    X_test  : test feature matrix
    k       : number of nearest neighbours

    Returns
    -------
    List of predicted class labels.
    """
    predictions = []
    for test_point in X_test.values:
        distances = [
            (euclidean_distance(test_point, X_train.iloc[i].values), y_train.iloc[i])
            for i in range(len(X_train))
        ]
        distances.sort(key=lambda x: x[0])
        neighbor_labels = [label for _, label in distances[:k]]
        predictions.append(max(set(neighbor_labels), key=neighbor_labels.count))
    return predictions

## 4. Algorithm 1 – Decision Tree

Three dataset sizes are used to observe how the tree scales:
- **Small** (10 000 samples, `max_depth=5`)
- **Medium** (50 000 samples, `max_depth=10`)
- **Full** dataset (unrestricted depth)

In [ ]:
X_full = df.drop("Class", axis=1)
y_full = df["Class"]

EXPERIMENTS = [
    ("Small (n=10 000)",  X_full.sample(n=10_000, random_state=RANDOM_STATE), 5),
    ("Medium (n=50 000)", X_full.sample(n=50_000, random_state=RANDOM_STATE), 10),
    ("Full dataset",      X_full,                                              None),
]

for label, X_exp, depth in EXPERIMENTS:
    y_exp = y_full.loc[X_exp.index]

    X_train, X_test, y_train, y_test = train_test_split(
        X_exp, y_exp, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print_evaluation(f"Decision Tree – {label}", y_test, y_pred)


  Decision Tree – Small (n=10 000)
Accuracy : 0.9993

Confusion Matrix:
[[1467    1]
 [   1 1531]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1468
           1       1.00      1.00      1.00      1532

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000


  Decision Tree – Medium (n=50 000)
Accuracy : 0.9995

Confusion Matrix:
[[7509    0]
 [   7 7484]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7509
           1       1.00      1.00      1.00      7491

    accuracy                           1.00     15000
   macro avg       1.00      1.00      1.00     15000
weighted avg       1.00      1.00      1.00     15000


  Decision Tree – Full dataset
Accuracy : 0.9996

Confusion Matrix:
[[85108    41]
 [   31 85409

## 5. Algorithm 2 – K-Nearest Neighbors (Custom Implementation)

Features are standardised before computing distances.  
Three dataset sizes are used: Small (2 000), Medium (5 000), and Full.

In [ ]:
scaler  = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_full), index=X_full.index)

KNN_EXPERIMENTS = [
    ("Small (n=2 000)",  X_scaled.sample(n=2_000,  random_state=RANDOM_STATE)),
    ("Medium (n=5 000)", X_scaled.sample(n=5_000,  random_state=RANDOM_STATE)),
    ("Full dataset",     X_scaled),
]

for label, X_exp in KNN_EXPERIMENTS:
    y_exp = y_full.loc[X_exp.index]

    X_train, X_test, y_train, y_test = train_test_split(
        X_exp, y_exp, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    y_pred = knn_predict(X_train, y_train, X_test, k=KNN_NEIGHBORS)
    print_evaluation(f"KNN – {label}", y_test, y_pred)


  KNN – Small (n=2 000)
Accuracy : 0.9933

Confusion Matrix:
[[310   1]
 [  3 286]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       311
           1       1.00      0.99      0.99       289

    accuracy                           0.99       600
   macro avg       0.99      0.99      0.99       600
weighted avg       0.99      0.99      0.99       600


  KNN – Medium (n=5 000)
Accuracy : 0.9933

Confusion Matrix:
[[745   6]
 [  4 745]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       751
           1       0.99      0.99      0.99       749

    accuracy                           0.99      1500
   macro avg       0.99      0.99      0.99      1500
weighted avg       0.99      0.99      0.99      1500



### 5.1 Case-Based Evaluation

| Case    | Description                              |
|---------|------------------------------------------|
| Best    | Perfectly balanced classes               |
| Average | Random sample (natural distribution)     |
| Worst   | Severely imbalanced (90 % normal)        |

In [ ]:
CASE_CONFIGS = {
    "Best Case":    sample_balanced(df, n=5000),
    "Average Case": df.sample(n=5000, random_state=RANDOM_STATE),
    "Worst Case":   sample_imbalanced(df, n=5000),
}

# ── Decision Tree ─────────────────────────────────────────────────────────
print("\n" + "─"*55)
print("  DECISION TREE – Case-Based Evaluation")
print("─"*55)

for case_name, sample_df in CASE_CONFIGS.items():
    X = sample_df.drop("Class", axis=1)
    y = sample_df["Class"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    depth = {"Best Case": 5, "Average Case": 10, "Worst Case": None}[case_name]
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    print_evaluation(f"Decision Tree – {case_name}", y_test, model.predict(X_test))

# ── KNN ───────────────────────────────────────────────────────────────────
print("\n" + "─"*55)
print("  KNN – Case-Based Evaluation")
print("─"*55)

for case_name, sample_df in CASE_CONFIGS.items():
    X = sample_df.drop("Class", axis=1)
    y = sample_df["Class"]
    X_sc = pd.DataFrame(StandardScaler().fit_transform(X))
    X_train, X_test, y_train, y_test = train_test_split(
        X_sc, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    y_pred = knn_predict(X_train, y_train, X_test, k=KNN_NEIGHBORS)
    print_evaluation(f"KNN – {case_name}", y_test, y_pred)


───────────────────────────────────────────────────────
  DECISION TREE – Case-Based Evaluation
───────────────────────────────────────────────────────

  Decision Tree – Best Case
Accuracy : 0.9987

Confusion Matrix:
[[730   0]
 [  2 768]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       730
           1       1.00      1.00      1.00       770

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500


  Decision Tree – Average Case
Accuracy : 0.9993

Confusion Matrix:
[[751   0]
 [  1 748]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       751
           1       1.00      1.00      1.00       749

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       

## 6. Running Time Experiments

Each algorithm is benchmarked across multiple input sizes and repeated **20 times** per size.  
Results include: individual run times, experimental average, theoretical estimate, and their ratio.

| Algorithm     | Theoretical Complexity | Estimate formula |
|---------------|------------------------|------------------|
| Decision Tree | O(n log n)             | n · log₂(n)      |
| KNN           | O(n)  (sklearn ball-tree prediction) | n |

In [ ]:
def build_timing_table(times_by_size: dict, theory_fn, repeats: int = REPEATS) -> pd.DataFrame:
    """
    Build a results DataFrame with individual times, averages,
    theoretical estimates, and the experimental/theory ratio.

    Parameters
    ----------
    times_by_size : {size: [t1, t2, ...]} mapping
    theory_fn     : callable(n) → theoretical estimate for input size n
    repeats       : number of timing runs per size
    """
    df_table = pd.DataFrame(times_by_size)
    df_table.index = [f"Run {i+1:02d} (ms)" for i in range(repeats)]

    exp_avgs   = df_table.mean()
    theoretics = pd.Series({n: theory_fn(n) for n in df_table.columns})
    ratios     = exp_avgs / theoretics

    df_table.loc["Experimental Average"] = exp_avgs
    df_table.loc["Theoretical Estimate"] = theoretics
    df_table.loc["Experimental/Theory"]  = ratios

    return df_table


def run_decision_tree_timing(data: pd.DataFrame, sizes: list,
                              repeats: int = REPEATS, case_type: str = "average") -> dict:
    """Time the Decision Tree across multiple input sizes."""
    results = {}
    for size in sizes:
        times = []
        for i in range(repeats):
            if case_type == "average":
                sample_df = data.sample(n=size, random_state=i)
            else:  # worst: imbalanced
                sample_df = sample_imbalanced(data, n=size, random_state=i)

            X = sample_df.drop("Class", axis=1)
            y = sample_df["Class"]
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=i
            )
            start = time.time()
            DecisionTreeClassifier(random_state=i).fit(X_train, y_train).predict(X_test)
            times.append((time.time() - start) * 1000)
        results[size] = times
    return results


def run_knn_timing(data: pd.DataFrame, sizes: list,
                   repeats: int = REPEATS, case_type: str = "average") -> dict:
    """Time the sklearn KNN across multiple input sizes."""
    results = {}
    for size in sizes:
        times = []
        for i in range(repeats):
            if case_type == "best":
                sample_df = sample_balanced(data, n=size, random_state=i)
            elif case_type == "worst":
                sample_df = sample_imbalanced(data, n=size, random_state=i)
            else:
                sample_df = data.sample(n=size, random_state=i)

            X = sample_df.drop("Class", axis=1)
            y = sample_df["Class"]
            X_sc = StandardScaler().fit_transform(X)
            X_train, X_test, y_train, y_test = train_test_split(
                X_sc, y, test_size=TEST_SIZE, random_state=i
            )
            start = time.time()
            KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS).fit(X_train, y_train).predict(X_test)
            times.append((time.time() - start) * 1000)
        results[size] = times
    return results

### 6.1 Decision Tree – Timing Results

In [ ]:
dt_theory = lambda n: n * math.log2(n)

dt_avg_small  = build_timing_table(run_decision_tree_timing(df, SMALL_SIZES, case_type="average"), dt_theory)
dt_avg_large  = build_timing_table(run_decision_tree_timing(df, LARGE_SIZES, case_type="average"), dt_theory)
dt_worst_small = build_timing_table(run_decision_tree_timing(df, SMALL_SIZES, case_type="worst"),  dt_theory)
dt_worst_large = build_timing_table(run_decision_tree_timing(df, LARGE_SIZES, case_type="worst"),  dt_theory)

In [ ]:
print("Decision Tree – Average Case, Small Inputs")
dt_avg_small

Decision Tree – Average Case, Small Inputs


,1000,2000,3000,4000
Run 01 (ms),9.368181,11.375189,21.229744,36.999464
Run 02 (ms),7.664442,11.534214,21.585703,36.956072
Run 03 (ms),7.157087,18.018961,21.559954,46.427250
Run 04 (ms),7.442236,14.943600,15.527010,26.354074
Run 05 (ms),7.514000,14.881849,21.341324,45.361519
Run 06 (ms),7.957220,15.291452,21.400452,40.235281
Run 07 (ms),9.259939,15.207767,16.299009,36.727905
Run 08 (ms),7.371664,15.385389,27.022362,36.893129
Run 09 (ms),7.344007,11.366367,27.842283,38.308620
Run 10 (ms),7.267952,15.394449,21.660566,37.754297


In [ ]:
print("Decision Tree – Average Case, Large Inputs")
dt_avg_large

Decision Tree – Average Case, Large Inputs


,10000,20000,30000,50000
Run 01 (ms),130.883932,351.515293,438.311338,915.816545
Run 02 (ms),153.128862,235.079050,470.736504,687.086105
Run 03 (ms),117.380142,267.098665,389.034271,785.804033
Run 04 (ms),89.881897,263.924122,462.210894,782.636404
Run 05 (ms),168.781519,402.095318,538.564920,1216.919184
Run 06 (ms),114.460707,214.469433,626.051664,1023.175478
Run 07 (ms),136.135101,381.739616,617.618084,1554.037094
Run 08 (ms),128.054142,446.410656,909.354448,1518.802643
Run 09 (ms),104.443550,349.097967,649.161816,1111.008644
Run 10 (ms),107.753515,264.389515,343.302250,781.905413


In [ ]:
print("Decision Tree – Worst Case, Small Inputs")
dt_worst_small

Decision Tree – Worst Case, Small Inputs


,1000,2000,3000,4000
Run 01 (ms),7.420301,11.653185,16.284704,20.022631
Run 02 (ms),7.274866,18.649101,26.825905,34.051180
Run 03 (ms),7.459641,11.564970,16.803980,19.606829
Run 04 (ms),7.094860,11.471510,16.245604,19.602537
Run 05 (ms),7.288933,11.600971,16.213417,19.964933
Run 06 (ms),10.826111,18.361568,26.934385,34.135818
Run 07 (ms),7.143021,11.560917,17.497540,20.228624
Run 08 (ms),7.492542,11.353493,16.359568,20.159960
Run 09 (ms),7.338524,11.461973,15.748501,36.693335
Run 10 (ms),8.463144,12.037277,16.090155,34.467459


In [ ]:
print("Decision Tree – Worst Case, Large Inputs")
dt_worst_large

Decision Tree – Worst Case, Large Inputs


,10000,20000,30000,50000
Run 01 (ms),46.148300,331.454992,379.623890,645.754576
Run 02 (ms),82.743645,434.224129,494.022369,842.881680
Run 03 (ms),46.353817,228.892326,249.228477,835.333347
Run 04 (ms),54.286957,351.233959,136.151314,848.990202
Run 05 (ms),119.428873,335.900784,256.097078,632.295132
Run 06 (ms),83.426952,227.735281,383.583069,1263.556719
Run 07 (ms),46.012402,110.262632,252.476931,635.002613
Run 08 (ms),82.053423,164.680243,379.632235,830.829859
Run 09 (ms),83.244801,242.691278,370.623112,650.273323
Run 10 (ms),119.142294,328.371286,497.135401,881.042957


### 6.2 KNN – Timing Results

In [ ]:
knn_theory = lambda n: n  # O(n) prediction with sklearn ball-tree

knn_best_small    = build_timing_table(run_knn_timing(df, SMALL_SIZES, case_type="best"),    knn_theory)
knn_best_large    = build_timing_table(run_knn_timing(df, LARGE_SIZES, case_type="best"),    knn_theory)
knn_avg_small     = build_timing_table(run_knn_timing(df, SMALL_SIZES, case_type="average"), knn_theory)
knn_avg_large     = build_timing_table(run_knn_timing(df, LARGE_SIZES, case_type="average"), knn_theory)
knn_worst_small   = build_timing_table(run_knn_timing(df, SMALL_SIZES, case_type="worst"),   knn_theory)
knn_worst_large   = build_timing_table(run_knn_timing(df, LARGE_SIZES, case_type="worst"),   knn_theory)

In [ ]:
print("KNN – Best Case, Small Inputs")
knn_best_small

KNN – Best Case, Small Inputs


,1000,2000,3000,4000
Run 01 (ms),61.488628,7.339001,24.999142,23.199797
Run 02 (ms),3.174543,7.314444,25.627613,22.812843
Run 03 (ms),3.192425,7.620335,24.876595,23.062706
Run 04 (ms),3.081560,7.362843,24.085999,23.119450
Run 05 (ms),3.174543,7.921457,24.054766,22.999525
Run 06 (ms),3.052235,7.505178,23.057938,22.922993
Run 07 (ms),3.130674,12.344360,27.095079,35.333872
Run 08 (ms),3.195763,11.891365,24.387598,22.997141
Run 09 (ms),3.243923,13.048649,24.565458,22.977114
Run 10 (ms),3.549337,13.298035,25.110245,22.837877


In [ ]:
print("KNN – Best Case, Large Inputs")
knn_best_large

KNN – Best Case, Large Inputs


,10000,20000,30000,50000
Run 01 (ms),130.897522,511.394262,1164.498329,4664.895296
Run 02 (ms),133.203983,513.246536,1167.291641,3183.284998
Run 03 (ms),130.122900,532.364607,2099.414110,3219.448566
Run 04 (ms),145.519733,514.826059,1573.083878,3825.920582
Run 05 (ms),130.977392,532.748938,1162.798643,3920.663834
Run 06 (ms),130.946398,889.850855,1179.302931,3243.200541
Run 07 (ms),131.640434,967.046738,1160.455227,3224.918365
Run 08 (ms),131.053686,931.211472,1155.726433,4648.533821
Run 09 (ms),149.761200,554.147720,1161.957026,3202.851772
Run 10 (ms),133.816719,540.046453,1167.986631,3225.308895


In [ ]:
print("KNN – Average Case, Small Inputs")
knn_avg_small

KNN – Average Case, Small Inputs


,1000,2000,3000,4000
Run 01 (ms),3.204107,7.362127,14.415741,23.093224
Run 02 (ms),3.088474,7.351398,14.207602,23.137569
Run 03 (ms),3.219843,7.352591,14.239311,23.063421
Run 04 (ms),3.129005,7.503748,13.992786,23.173809
Run 05 (ms),3.103971,7.533550,14.616251,22.935629
Run 06 (ms),3.093719,7.368326,14.689684,24.199009
Run 07 (ms),3.246069,7.390976,17.275572,24.039745
Run 08 (ms),3.103018,7.361650,14.147520,23.067474
Run 09 (ms),3.214598,7.353067,16.505480,24.088860
Run 10 (ms),3.108740,7.605076,14.211893,25.375843


In [ ]:
print("KNN – Average Case, Large Inputs")
knn_avg_large

KNN – Average Case, Large Inputs


,10000,20000,30000,50000
Run 01 (ms),130.921364,529.767752,1444.957733,3204.112291
Run 02 (ms),131.038189,514.619589,1162.443638,3200.578213
Run 03 (ms),132.978916,526.752710,1179.714441,3508.334398
Run 04 (ms),147.145271,515.206099,1157.602310,4293.035269
Run 05 (ms),132.763863,544.095755,1161.836386,3252.779722
Run 06 (ms),134.200096,528.923988,1193.147421,3227.246284
Run 07 (ms),130.920887,511.975765,1159.547567,4672.696114
Run 08 (ms),131.880045,532.440424,1162.503481,3215.731859
Run 09 (ms),135.686159,515.911579,1654.895306,3221.257925
Run 10 (ms),251.403570,539.660215,2070.676327,3271.138191


In [ ]:
print("KNN – Worst Case, Small Inputs")
knn_worst_small

KNN – Worst Case, Small Inputs


,1000,2000,3000,4000
Run 01 (ms),5.003691,11.042118,14.124155,23.466110
Run 02 (ms),5.433083,12.419939,14.767647,23.281097
Run 03 (ms),5.072832,12.398005,14.483690,23.622513
Run 04 (ms),5.019903,12.335539,16.493082,23.512602
Run 05 (ms),5.394697,12.245178,14.432192,23.209333
Run 06 (ms),5.029202,12.725830,14.308929,23.262262
Run 07 (ms),5.002737,12.742519,14.663935,23.788452
Run 08 (ms),5.022049,12.478590,14.999151,23.045778
Run 09 (ms),5.086660,7.672310,14.338493,35.812616
Run 10 (ms),4.958630,7.525444,14.662981,23.536205


In [ ]:
print("KNN – Worst Case, Large Inputs")
knn_worst_large

KNN – Worst Case, Large Inputs


,10000,20000,30000,50000
Run 01 (ms),132.674694,520.568609,2154.255629,3224.152565
Run 02 (ms),132.476091,931.744814,1565.809965,3249.842405
Run 03 (ms),134.589672,962.626934,1160.333872,3657.699585
Run 04 (ms),134.598017,917.816401,1206.140757,4165.946007
Run 05 (ms),132.374763,531.967163,1173.682213,3252.717972
Run 06 (ms),136.969328,538.875103,1168.968678,3241.671085
Run 07 (ms),132.863283,518.671513,1171.541452,4624.674797
Run 08 (ms),135.810137,540.602684,1173.140764,3206.722736
Run 09 (ms),142.760038,512.646914,1172.513008,3222.920179
Run 10 (ms),132.660627,516.713619,2107.686281,3607.455492
